# Frankenstein MPC notebook

This notebook merges the **building RC model**, the **heat pump model**, and the **thermal storage tank model** into a single control-oriented workflow.

## Current modelling choices
- **Building**: the original **3R2C** structure is preserved as much as possible.
- **Heat pump**: quasi-steady control-oriented model with **capacity** and **COP** maps.
- **Tank**: **3-node flow-based** stratified storage model.
- **Architecture**: **HP → Tank → Building**.
- **Control**: **economic MPC** with:
  - indoor comfort constraints,
  - slack variables for feasibility,
  - electricity cost based on Italian **F1/F2/F3** time bands,
  - a terminal tank penalty to avoid using the initial storage content as free energy,
  - small smoothing penalties on flow-rate control actions.

## Notes
This version uses a more physical HP–tank–building coupling than the first power-based build. The heat pump charges the tank through an equivalent charging flow, while the building extracts heat through an equivalent load-side flow.


In [ ]:
import importlib
import subprocess
import sys

required_packages = ["casadi"]
for package in required_packages:
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])


In [ ]:
from dataclasses import dataclass
from pathlib import Path

import casadi as ca
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.linalg import expm


## Configuration

The baseline sizing used here is:
- nominal heat pump thermal capacity: **10 kW**
- tank size: **25 L/kW**
- resulting tank volume: **250 L**

The electricity prices are represented with fixed year-round values by time band:
- **F1 = 0.3291 €/kWh**
- **F2 = 0.3439 €/kWh**
- **F3 = 0.3060 €/kWh**

The notebook automatically uses `data.csv` if it exists in the repository root.  
If no external dataset is found, a synthetic weather profile is generated so that the notebook remains runnable.


In [ ]:
# -----------------------------
# Global configuration
# -----------------------------
DT_HOURS = 0.25                    # 15 minutes
SIMULATION_HOURS = 72              # closed-loop simulation length
PREDICTION_HOURS = 6               # MPC horizon
HORIZON_STEPS = int(PREDICTION_HOURS / DT_HOURS)

PRICE_BY_BAND_EUR_PER_KWH = {
    "F1": 0.3291,
    "F2": 0.3439,
    "F3": 0.3060,
}

# Building RC model parameters (preserved from build_RC_MPC as much as possible)
BUILDING_PARAMS = {
    "Rea": 58.4895,   # [°C/kW]
    "Rie": 1.1521,    # [°C/kW]
    "Rinf": 15.748,   # [°C/kW]
    "Ce": 3.5,        # [kWh/°C]
    "Ci": 1.15,       # [kWh/°C]
    "Ai": 1.56,       # [m²] effective solar aperture
    "Ae": 0.122,      # [m²] effective solar aperture
    "heating_efficiency": 1.0,
}

# Heat pump and tank baseline sizing
P_HP_NOMINAL_KW = 10.0
TANK_VOLUME_M3 = 25.0 / 1000.0 * P_HP_NOMINAL_KW  # 25 L/kW

# Flow-based tank / hydronic assumptions
T_TANK_PHYSICAL_MIN_C = 30.0
T_TANK_MAX_C = 60.0

T_HP_SUPPLY_C = 50.0          # HP supply temperature to tank [°C]
T_LOAD_RETURN_C = 30.0        # simplified return temperature from fan-coil/building [°C]
T_SUPPLY_MIN_C = 40.0         # minimum useful top-node temperature for heat delivery [°C]

M_DOT_HP_MAX_KG_S = 0.30
M_DOT_LOAD_MAX_KG_S = 0.20
Q_BUILD_MAX_KW = 6.0

# Terminal tank target: [bottom, middle, top]
T_TANK_TARGET_PROFILE_C = np.array([35.0, 40.0, 45.0], dtype=float)
TANK_TERMINAL_WEIGHT = 0.20

# Control penalties
TRACK_WEIGHT = 1.0
SLACK_WEIGHT = 1000.0
SMOOTH_WEIGHT = 0.02

# Comfort band
TI_MIN_C = 20.0
TI_MAX_C = 24.0
TI_REF_C = 22.0


In [ ]:
# -----------------------------
# Utilities
# -----------------------------
def discretize_linear_system(A: np.ndarray, B: np.ndarray, dt_hours: float) -> tuple[np.ndarray, np.ndarray]:
    """Discretize a continuous-time linear state-space model."""
    A = np.asarray(A, dtype=float)
    B = np.asarray(B, dtype=float)
    A_d = expm(A * dt_hours)
    B_d = np.linalg.solve(A, (A_d - np.eye(A.shape[0]))) @ B
    return A_d, B_d


def get_arera_time_band(timestamp: pd.Timestamp) -> str:
    """Return F1/F2/F3 using the standard Italian calendar-based time bands."""
    weekday = timestamp.weekday()  # Monday = 0
    hour = timestamp.hour + timestamp.minute / 60.0

    # Public holidays are ignored in this v1 and can be added later.
    if weekday <= 4:  # Monday-Friday
        if 8 <= hour < 19:
            return "F1"
        if (7 <= hour < 8) or (19 <= hour < 23):
            return "F2"
        return "F3"

    if weekday == 5:  # Saturday
        return "F2" if 7 <= hour < 23 else "F3"

    return "F3"  # Sunday


def load_or_generate_weather_data(
    dt_hours: float = DT_HOURS,
    simulation_hours: float = SIMULATION_HOURS,
    csv_path: str | Path = "data.csv",
) -> pd.DataFrame:
    """
    Load external inputs from data.csv if available.
    Required columns when reading an external file:
    - Ta : outdoor air temperature [°C]
    - Gv : global horizontal or effective solar signal [kW/m² equivalent input used by the RC model]

    Optional:
    - timestamp : datetime-like column
    """
    csv_path = Path(csv_path)

    if csv_path.exists():
        df = pd.read_csv(csv_path)

        if "Ta" not in df.columns or "Gv" not in df.columns:
            raise ValueError("The external dataset must contain at least 'Ta' and 'Gv' columns.")

        n_steps = int(simulation_hours / dt_hours) + HORIZON_STEPS + 1
        df = df.iloc[:n_steps].copy().reset_index(drop=True)

        if "timestamp" in df.columns:
            df["timestamp"] = pd.to_datetime(df["timestamp"])
        else:
            df["timestamp"] = pd.date_range(
                "2023-01-02 00:00:00",
                periods=len(df),
                freq=f"{int(dt_hours * 60)}min",
            )

        df["Ta"] = df["Ta"].astype(float)
        df["Gv"] = df["Gv"].astype(float)
        source = "external data.csv"
    else:
        n_steps = int(simulation_hours / dt_hours) + HORIZON_STEPS + 1
        t_hours = np.arange(n_steps) * dt_hours

        df = pd.DataFrame({
            "timestamp": pd.date_range(
                "2023-01-02 00:00:00",
                periods=n_steps,
                freq=f"{int(dt_hours * 60)}min",
            )
        })

        # Synthetic but smooth profile for a winter-like test
        df["Ta"] = 6.0 + 4.0 * np.sin(2.0 * np.pi * (t_hours - 8.0) / 24.0)
        df["Gv"] = np.maximum(0.0, 0.6 * np.sin(2.0 * np.pi * (t_hours - 6.0) / 24.0))
        source = "synthetic fallback profile"

    df["band"] = df["timestamp"].apply(get_arera_time_band)
    df["price_eur_per_kwh"] = df["band"].map(PRICE_BY_BAND_EUR_PER_KWH)

    print(f"Weather / price source: {source}")
    print(f"Rows available: {len(df)}")
    return df


df_inputs = load_or_generate_weather_data()
df_inputs.head()


## Building model

The building model keeps the original **3R2C** formulation and its parameter set.  
The only conceptual change is that the thermal input is now interpreted as:

\[
Q_{build}
\]

that is, the **thermal power effectively delivered to the building from the tank**, not directly from the heat pump.


In [ ]:
# -----------------------------
# Building RC model
# -----------------------------
@dataclass
class BuildingRCParameters:
    Rea: float
    Rie: float
    Rinf: float
    Ce: float
    Ci: float
    Ai: float
    Ae: float
    heating_efficiency: float = 1.0


class BuildingRCModel:
    def __init__(self, params: BuildingRCParameters, dt_hours: float) -> None:
        self.params = params
        self.dt_hours = dt_hours

        A_c = np.array([
            [
                -1.0 / (params.Ci * params.Rie) - 1.0 / (params.Ci * params.Rinf),
                1.0 / (params.Ci * params.Rie),
            ],
            [
                1.0 / (params.Ce * params.Rie),
                -1.0 / (params.Ce * params.Rie) - 1.0 / (params.Ce * params.Rea),
            ],
        ])

        B_c = np.array([
            [1.0 / params.Ci, 1.0 / (params.Ci * params.Rinf), params.Ai / params.Ci],
            [0.0, 1.0 / (params.Ce * params.Rea), params.Ae / params.Ce],
        ])

        self.A_d, self.B_d = discretize_linear_system(A_c, B_c, dt_hours)

    def step_numpy(self, x_k: np.ndarray, q_build_kw: float, t_out_c: float, g_solar: float) -> np.ndarray:
        u_k = np.array([
            q_build_kw * self.params.heating_efficiency,
            t_out_c,
            g_solar,
        ])
        return self.A_d @ x_k + self.B_d @ u_k


building_model = BuildingRCModel(
    BuildingRCParameters(**BUILDING_PARAMS),
    dt_hours=DT_HOURS,
)

print("Discrete A matrix:")
print(np.round(building_model.A_d, 4))
print("\nDiscrete B matrix:")
print(np.round(building_model.B_d, 4))


## Heat pump model

The heat pump is modeled in a quasi-steady way through:
- a **capacity map**,
- a **COP map**.

In this flow-based version, the sink-side temperature used by the HP map is the imposed heat-pump supply temperature to the tank, `T_HP_SUPPLY_C`.  
This avoids using the tank top node as both a storage state and a direct HP supply target.


In [ ]:
# -----------------------------
# Heat pump model
# -----------------------------
@dataclass
class QuasiSteadyHeatPumpParameters:
    q_ref_kw: float
    cop_ref: float
    t_source_ref_c: float
    t_sink_ref_c: float
    alpha_q_src: float
    alpha_q_sink: float
    alpha_cop_src: float
    alpha_cop_sink: float


HP_PARAMS = QuasiSteadyHeatPumpParameters(
    q_ref_kw=P_HP_NOMINAL_KW,
    cop_ref=3.5,
    t_source_ref_c=7.0,
    t_sink_ref_c=35.0,
    alpha_q_src=0.20,
    alpha_q_sink=-0.08,
    alpha_cop_src=0.08,
    alpha_cop_sink=-0.04,
)


def hp_capacity_kw(t_source_c, t_sink_c, params: QuasiSteadyHeatPumpParameters):
    return (
        params.q_ref_kw
        + params.alpha_q_src * (t_source_c - params.t_source_ref_c)
        + params.alpha_q_sink * (t_sink_c - params.t_sink_ref_c)
    )


def hp_cop(t_source_c, t_sink_c, params: QuasiSteadyHeatPumpParameters):
    return (
        params.cop_ref
        + params.alpha_cop_src * (t_source_c - params.t_source_ref_c)
        + params.alpha_cop_sink * (t_sink_c - params.t_sink_ref_c)
    )


# Quick sanity check
print("Example HP operating point")
print("Q_max [kW]:", round(float(hp_capacity_kw(5.0, 40.0, HP_PARAMS)), 3))
print("COP [-]:", round(float(hp_cop(5.0, 40.0, HP_PARAMS)), 3))


## Flow-based 3-node tank model

The tank is represented with three vertical nodes:

\[
T_{bottom}, \quad T_{middle}, \quad T_{top}
\]

The architecture is:

\[
HP ightarrow Tank ightarrow Building
\]

but the coupling is now flow-based rather than purely power-based.

### Heat pump side
The heat pump draws water from the bottom region and injects heated water at the top region.  
This is represented by an equivalent charging mass flow rate:

\[
\dot m_{hp}
\]

and a fixed HP supply temperature:

\[
T_{hp,supply}
\]

### Building/load side
The building draws hot water from the tank top and returns cooler water to the bottom.  
This is represented by an equivalent load-side mass flow rate:

\[
\dot m_{load}
\]

and a simplified return temperature:

\[
T_{load,return}
\]

The heat delivered to the building is therefore derived from the tank state:

\[
Q_{build} = \dot m_{load} c_p (T_{top} - T_{load,return})
\]

The model also includes inter-node conductance, ambient losses, and smooth anti-inversion mixing.


In [ ]:
# -----------------------------
# 3-node flow-based tank model
# -----------------------------
@dataclass
class ThreeNodeTankParameters:
    volume_m3: float
    ua_total_kw_per_k: float
    g_between_nodes_kw_per_k: float
    t_min_c: float
    t_max_c: float
    t_hp_supply_c: float
    t_load_return_c: float
    t_supply_min_c: float
    m_dot_hp_max_kg_s: float
    m_dot_load_max_kg_s: float
    rho_kg_per_m3: float = 1000.0
    cp_kwh_per_kgk: float = 4180.0 / 3600.0 / 1000.0
    inversion_mixing_fraction: float = 0.10
    inversion_softplus_beta: float = 10.0


class ThreeNodeTankModel:
    """
    Three-node flow-based stratified tank.

    State ordering:
    - x[0] = bottom temperature [°C]
    - x[1] = middle temperature [°C]
    - x[2] = top temperature [°C]

    Decision variables:
    - m_dot_hp_kg_s: equivalent HP charging flow rate [kg/s]
    - m_dot_load_kg_s: equivalent load-side flow rate [kg/s]
    """

    def __init__(self, params: ThreeNodeTankParameters, dt_hours: float) -> None:
        self.params = params
        self.dt_hours = dt_hours

        c_total = params.rho_kg_per_m3 * params.volume_m3 * params.cp_kwh_per_kgk
        self.c_node_kwh_per_k = c_total / 3.0
        self.ua_node_kw_per_k = params.ua_total_kw_per_k / 3.0
        self.g_between_nodes_kw_per_k = params.g_between_nodes_kw_per_k

    @property
    def cp_flow_kw_per_kg_s_k(self) -> float:
        """
        Heat capacity for flow power calculations [kW / (kg/s K)].
        """
        return self.params.cp_kwh_per_kgk * 3600.0

    def q_hp_to_tank_kw_numpy(self, x_tank_c: np.ndarray, m_dot_hp_kg_s: float) -> float:
        """
        Thermal power transferred by HP into the tank [kW].
        Based on bottom-node suction and fixed HP supply temperature.
        """
        T_bottom = float(x_tank_c[0])
        return m_dot_hp_kg_s * self.cp_flow_kw_per_kg_s_k * (self.params.t_hp_supply_c - T_bottom)

    def q_tank_to_building_kw_numpy(self, x_tank_c: np.ndarray, m_dot_load_kg_s: float) -> float:
        """
        Thermal power delivered from tank to building [kW].
        Based on top-node supply and fixed load return temperature.
        """
        T_top = float(x_tank_c[2])
        return m_dot_load_kg_s * self.cp_flow_kw_per_kg_s_k * (T_top - self.params.t_load_return_c)

    def q_hp_to_tank_kw_casadi(self, x_tank_c, m_dot_hp_kg_s):
        T_bottom = x_tank_c[0]
        return m_dot_hp_kg_s * self.cp_flow_kw_per_kg_s_k * (self.params.t_hp_supply_c - T_bottom)

    def q_tank_to_building_kw_casadi(self, x_tank_c, m_dot_load_kg_s):
        T_top = x_tank_c[2]
        return m_dot_load_kg_s * self.cp_flow_kw_per_kg_s_k * (T_top - self.params.t_load_return_c)

    def _softplus_numpy(self, x: float | np.ndarray) -> float | np.ndarray:
        beta = self.params.inversion_softplus_beta
        xb = np.clip(beta * x, -50.0, 50.0)
        return np.log1p(np.exp(xb)) / beta

    def _softplus_casadi(self, x):
        beta = self.params.inversion_softplus_beta
        return ca.log(1.0 + ca.exp(beta * x)) / beta

    def smooth_inversion_mixing_numpy(self, temperatures_c: np.ndarray) -> np.ndarray:
        """
        Smooth local anti-inversion mixing term [K/h].
        """
        T = np.asarray(temperatures_c, dtype=float)

        eta = self.params.inversion_mixing_fraction
        if not (0.0 <= eta <= 1.0):
            raise ValueError("inversion_mixing_fraction must be between 0 and 1.")

        gamma_per_h = eta / (2.0 * self.dt_hours)

        s01 = self._softplus_numpy(T[0] - T[1])
        s12 = self._softplus_numpy(T[1] - T[2])

        return np.array([
            -gamma_per_h * s01,
             gamma_per_h * s01 - gamma_per_h * s12,
             gamma_per_h * s12,
        ])

    def smooth_inversion_mixing_casadi(self, temperatures_c):
        """
        CasADi equivalent of the smooth local anti-inversion mixing term [K/h].
        """
        T = temperatures_c

        eta = self.params.inversion_mixing_fraction
        gamma_per_h = eta / (2.0 * self.dt_hours)

        s01 = self._softplus_casadi(T[0] - T[1])
        s12 = self._softplus_casadi(T[1] - T[2])

        return ca.vertcat(
            -gamma_per_h * s01,
             gamma_per_h * s01 - gamma_per_h * s12,
             gamma_per_h * s12,
        )

    def rhs_numpy(
        self,
        x_tank_c: np.ndarray,
        m_dot_hp_kg_s: float,
        m_dot_load_kg_s: float,
        t_amb_c: float,
    ) -> np.ndarray:
        """
        Continuous-time right-hand side dT/dt [K/h].
        """
        T = np.asarray(x_tank_c, dtype=float)
        T_b, T_m, T_t = T

        cp_flow = self.cp_flow_kw_per_kg_s_k
        C_node = self.c_node_kwh_per_k
        g = self.g_between_nodes_kw_per_k
        ua = self.ua_node_kw_per_k

        # HP charging loop: hot water injected at top, equivalent downward displacement.
        q_hp_bottom = m_dot_hp_kg_s * cp_flow * (T_m - T_b)
        q_hp_middle = m_dot_hp_kg_s * cp_flow * (T_t - T_m)
        q_hp_top = m_dot_hp_kg_s * cp_flow * (self.params.t_hp_supply_c - T_t)

        # Load loop: hot water extracted from top, cooler return enters bottom.
        q_load_bottom = m_dot_load_kg_s * cp_flow * (self.params.t_load_return_c - T_b)
        q_load_middle = m_dot_load_kg_s * cp_flow * (T_b - T_m)
        q_load_top = m_dot_load_kg_s * cp_flow * (T_m - T_t)

        q_flow = np.array([
            q_hp_bottom + q_load_bottom,
            q_hp_middle + q_load_middle,
            q_hp_top + q_load_top,
        ])

        # Inter-node conductance.
        q_cond = np.array([
            g * (T_m - T_b),
            g * (T_b - T_m) + g * (T_t - T_m),
            g * (T_m - T_t),
        ])

        # Ambient losses, positive if ambient warms the node.
        q_loss = ua * (t_amb_c - T)

        dTdt = (q_flow + q_cond + q_loss) / C_node
        dTdt += self.smooth_inversion_mixing_numpy(T)

        return dTdt

    def rhs_casadi(
        self,
        x_tank_c,
        m_dot_hp_kg_s,
        m_dot_load_kg_s,
        t_amb_c,
    ):
        """
        CasADi-compatible continuous-time right-hand side dT/dt [K/h].
        """
        T_b = x_tank_c[0]
        T_m = x_tank_c[1]
        T_t = x_tank_c[2]

        cp_flow = self.cp_flow_kw_per_kg_s_k
        C_node = self.c_node_kwh_per_k
        g = self.g_between_nodes_kw_per_k
        ua = self.ua_node_kw_per_k

        q_hp_bottom = m_dot_hp_kg_s * cp_flow * (T_m - T_b)
        q_hp_middle = m_dot_hp_kg_s * cp_flow * (T_t - T_m)
        q_hp_top = m_dot_hp_kg_s * cp_flow * (self.params.t_hp_supply_c - T_t)

        q_load_bottom = m_dot_load_kg_s * cp_flow * (self.params.t_load_return_c - T_b)
        q_load_middle = m_dot_load_kg_s * cp_flow * (T_b - T_m)
        q_load_top = m_dot_load_kg_s * cp_flow * (T_m - T_t)

        q_flow = ca.vertcat(
            q_hp_bottom + q_load_bottom,
            q_hp_middle + q_load_middle,
            q_hp_top + q_load_top,
        )

        q_cond = ca.vertcat(
            g * (T_m - T_b),
            g * (T_b - T_m) + g * (T_t - T_m),
            g * (T_m - T_t),
        )

        q_loss = ca.vertcat(
            ua * (t_amb_c - T_b),
            ua * (t_amb_c - T_m),
            ua * (t_amb_c - T_t),
        )

        dTdt = (q_flow + q_cond + q_loss) / C_node
        dTdt += self.smooth_inversion_mixing_casadi(x_tank_c)

        return dTdt

    def step_numpy(
        self,
        x_k: np.ndarray,
        m_dot_hp_kg_s: float,
        m_dot_load_kg_s: float,
        t_amb_c: float,
    ) -> np.ndarray:
        """
        Forward-Euler plant step.
        """
        dTdt = self.rhs_numpy(
            x_tank_c=x_k,
            m_dot_hp_kg_s=m_dot_hp_kg_s,
            m_dot_load_kg_s=m_dot_load_kg_s,
            t_amb_c=t_amb_c,
        )
        x_next = np.asarray(x_k, dtype=float) + self.dt_hours * dTdt
        return np.clip(x_next, self.params.t_min_c, self.params.t_max_c)

    def step_casadi(
        self,
        x_k,
        m_dot_hp_kg_s,
        m_dot_load_kg_s,
        t_amb_c,
    ):
        """
        CasADi-compatible forward-Euler tank step used in the MPC prediction model.
        """
        dTdt = self.rhs_casadi(
            x_tank_c=x_k,
            m_dot_hp_kg_s=m_dot_hp_kg_s,
            m_dot_load_kg_s=m_dot_load_kg_s,
            t_amb_c=t_amb_c,
        )
        return x_k + self.dt_hours * dTdt


TANK_PARAMS = ThreeNodeTankParameters(
    volume_m3=TANK_VOLUME_M3,
    ua_total_kw_per_k=0.002,
    g_between_nodes_kw_per_k=0.08,
    t_min_c=T_TANK_PHYSICAL_MIN_C,
    t_max_c=T_TANK_MAX_C,
    t_hp_supply_c=T_HP_SUPPLY_C,
    t_load_return_c=T_LOAD_RETURN_C,
    t_supply_min_c=T_SUPPLY_MIN_C,
    m_dot_hp_max_kg_s=M_DOT_HP_MAX_KG_S,
    m_dot_load_max_kg_s=M_DOT_LOAD_MAX_KG_S,
    inversion_mixing_fraction=0.10,
    inversion_softplus_beta=10.0,
)

tank_model = ThreeNodeTankModel(TANK_PARAMS, dt_hours=DT_HOURS)

print(f"Tank volume [m³]: {TANK_VOLUME_M3:.3f}")
print(f"Node capacity [kWh/K]: {tank_model.c_node_kwh_per_k:.4f}")
print(f"cp for flow calculations [kW/(kg/s K)]: {tank_model.cp_flow_kw_per_kg_s_k:.3f}")
print(f"Physical tank bounds [°C]: {tank_model.params.t_min_c:.1f} -> {tank_model.params.t_max_c:.1f}")
print(f"Minimum useful top-node temperature [°C]: {tank_model.params.t_supply_min_c:.1f}")


## Economic MPC formulation

The controller uses:
- **building states**: indoor and envelope temperatures,
- **tank states**: bottom, middle, top temperatures,
- **decision variables**:
  - `M_dot_hp[k]`: equivalent HP charging mass flow rate,
  - `M_dot_load[k]`: equivalent tank-to-building mass flow rate.

The heat flows are now derived quantities:

\[
Q_{hp	o tank,k} = \dot m_{hp,k} c_p (T_{hp,supply} - T_{bottom,k})
\]

\[
Q_{tank	o building,k} = \dot m_{load,k} c_p (T_{top,k} - T_{load,return})
\]

The building RC model still receives only:

\[
Q_{build,k} = Q_{tank	o building,k}
\]

### Objective
At each step, the MPC minimizes:
1. electricity cost of the heat pump,
2. indoor temperature tracking penalty,
3. comfort slack penalties,
4. small smoothing penalties on flow-rate variations,
5. a **terminal tank penalty** to avoid emptying the initial thermal storage content for free.

### Important update
The storage is now physically coupled to the building demand through the top-node temperature and load-side flow rate. The tank can physically reach 30 °C, but useful delivery requires the top node to stay above the fan-coil supply threshold.


In [ ]:
# -----------------------------
# MPC builder
# -----------------------------
def build_economic_mpc_solver(
    building_model: BuildingRCModel,
    tank_model: ThreeNodeTankModel,
    hp_params: QuasiSteadyHeatPumpParameters,
    horizon_steps: int,
):
    opti = ca.Opti()

    # States
    X_build = opti.variable(2, horizon_steps + 1)  # [Ti, Te]
    X_tank = opti.variable(3, horizon_steps + 1)   # [T_bottom, T_middle, T_top]

    # Controls: equivalent hydronic flow rates
    M_dot_hp = opti.variable(1, horizon_steps)
    M_dot_load = opti.variable(1, horizon_steps)

    # Soft comfort constraints
    eps_low = opti.variable(1, horizon_steps)
    eps_high = opti.variable(1, horizon_steps)

    # Parameters / forecasts
    x0_build = opti.parameter(2)
    x0_tank = opti.parameter(3)
    ta_forecast = opti.parameter(1, horizon_steps)
    gh_forecast = opti.parameter(1, horizon_steps)
    price_forecast = opti.parameter(1, horizon_steps)

    opti.subject_to(X_build[:, 0] == x0_build)
    opti.subject_to(X_tank[:, 0] == x0_tank)

    objective = 0

    A_b = ca.DM(building_model.A_d)
    B_b = ca.DM(building_model.B_d)

    tank_target = ca.DM(T_TANK_TARGET_PROFILE_C)

    for k in range(horizon_steps):
        # Derived thermal powers
        q_hp_to_tank = tank_model.q_hp_to_tank_kw_casadi(
            x_tank_c=X_tank[:, k],
            m_dot_hp_kg_s=M_dot_hp[0, k],
        )
        q_tank_to_building = tank_model.q_tank_to_building_kw_casadi(
            x_tank_c=X_tank[:, k],
            m_dot_load_kg_s=M_dot_load[0, k],
        )

        # Building dynamics: original 3R2C structure, with tank-delivered heat input
        u_build = ca.vertcat(
            q_tank_to_building * building_model.params.heating_efficiency,
            ta_forecast[0, k],
            gh_forecast[0, k],
        )
        x_build_next = A_b @ X_build[:, k] + B_b @ u_build
        opti.subject_to(X_build[:, k + 1] == x_build_next)

        # Flow-based tank dynamics
        x_tank_next = tank_model.step_casadi(
            x_k=X_tank[:, k],
            m_dot_hp_kg_s=M_dot_hp[0, k],
            m_dot_load_kg_s=M_dot_load[0, k],
            t_amb_c=ta_forecast[0, k],
        )
        opti.subject_to(X_tank[:, k + 1] == x_tank_next)

        # HP algebraic maps using fixed HP supply temperature as sink-side condition
        q_hp_max = hp_capacity_kw(ta_forecast[0, k], tank_model.params.t_hp_supply_c, hp_params)
        cop_k = hp_cop(ta_forecast[0, k], tank_model.params.t_hp_supply_c, hp_params)

        # Flow bounds
        opti.subject_to(M_dot_hp[0, k] >= 0.0)
        opti.subject_to(M_dot_hp[0, k] <= tank_model.params.m_dot_hp_max_kg_s)
        opti.subject_to(M_dot_load[0, k] >= 0.0)
        opti.subject_to(M_dot_load[0, k] <= tank_model.params.m_dot_load_max_kg_s)

        # Derived heat-flow bounds
        opti.subject_to(q_hp_to_tank >= 0.0)
        opti.subject_to(q_hp_to_tank <= q_hp_max)
        opti.subject_to(q_tank_to_building >= 0.0)
        opti.subject_to(q_tank_to_building <= Q_BUILD_MAX_KW)

        # Tank state bounds
        opti.subject_to(X_tank[:, k] >= tank_model.params.t_min_c)
        opti.subject_to(X_tank[:, k] <= tank_model.params.t_max_c)

        # Minimum useful supply temperature from the top node
        opti.subject_to(X_tank[2, k] >= tank_model.params.t_supply_min_c)

        # Comfort soft constraints
        opti.subject_to(eps_low[0, k] >= 0.0)
        opti.subject_to(eps_high[0, k] >= 0.0)
        opti.subject_to(X_build[0, k] + eps_low[0, k] >= TI_MIN_C)
        opti.subject_to(X_build[0, k] - eps_high[0, k] <= TI_MAX_C)

        # Stage cost
        p_hp_el_kw = q_hp_to_tank / cop_k
        objective += price_forecast[0, k] * p_hp_el_kw * DT_HOURS
        objective += TRACK_WEIGHT * (X_build[0, k] - TI_REF_C) ** 2
        objective += SLACK_WEIGHT * (eps_low[0, k] ** 2 + eps_high[0, k] ** 2)

        # Flow-rate smoothing
        if k > 0:
            objective += SMOOTH_WEIGHT * (
                (M_dot_hp[0, k] - M_dot_hp[0, k - 1]) ** 2
                + (M_dot_load[0, k] - M_dot_load[0, k - 1]) ** 2
            )

    # Terminal tank bounds and terminal storage value
    opti.subject_to(X_tank[:, -1] >= tank_model.params.t_min_c)
    opti.subject_to(X_tank[:, -1] <= tank_model.params.t_max_c)
    opti.subject_to(X_tank[2, -1] >= tank_model.params.t_supply_min_c)

    terminal_error = X_tank[:, -1] - tank_target
    objective += TANK_TERMINAL_WEIGHT * ca.sumsqr(terminal_error)

    opti.minimize(objective)

    # Initial guesses improve robustness for the nonlinear flow-based formulation.
    opti.set_initial(X_build[0, :], TI_REF_C)
    opti.set_initial(X_build[1, :], TI_REF_C)
    opti.set_initial(X_tank[0, :], T_TANK_TARGET_PROFILE_C[0])
    opti.set_initial(X_tank[1, :], T_TANK_TARGET_PROFILE_C[1])
    opti.set_initial(X_tank[2, :], T_TANK_TARGET_PROFILE_C[2])
    opti.set_initial(M_dot_hp, 0.05)
    opti.set_initial(M_dot_load, 0.05)

    opti.solver(
        "ipopt",
        {"print_time": False},
        {"print_level": 0, "max_iter": 300},
    )

    return opti, {
        "X_build": X_build,
        "X_tank": X_tank,
        "M_dot_hp": M_dot_hp,
        "M_dot_load": M_dot_load,
        "eps_low": eps_low,
        "eps_high": eps_high,
        "x0_build": x0_build,
        "x0_tank": x0_tank,
        "ta_forecast": ta_forecast,
        "gh_forecast": gh_forecast,
        "price_forecast": price_forecast,
    }


opti, mpc_vars = build_economic_mpc_solver(
    building_model=building_model,
    tank_model=tank_model,
    hp_params=HP_PARAMS,
    horizon_steps=HORIZON_STEPS,
)

print(f"MPC horizon steps: {HORIZON_STEPS}")


In [ ]:
# -----------------------------
# Closed-loop simulation
# -----------------------------
def run_closed_loop_simulation(
    df: pd.DataFrame,
    building_model: BuildingRCModel,
    tank_model: ThreeNodeTankModel,
    hp_params: QuasiSteadyHeatPumpParameters,
    opti,
    mpc_vars,
):
    # Initial conditions
    x_build = np.array([21.5, 21.5], dtype=float)      # [Ti, Te]
    x_tank = np.array([35.0, 40.0, 45.0], dtype=float) # [bottom, middle, top]

    records = []

    n_steps = len(df) - 1 - HORIZON_STEPS
    for k in range(n_steps):
        horizon_slice = slice(k, k + HORIZON_STEPS)

        ta_fc = df["Ta"].iloc[horizon_slice].to_numpy(dtype=float).reshape(1, -1)
        gh_fc = df["Gv"].iloc[horizon_slice].to_numpy(dtype=float).reshape(1, -1)
        price_fc = df["price_eur_per_kwh"].iloc[horizon_slice].to_numpy(dtype=float).reshape(1, -1)

        opti.set_value(mpc_vars["x0_build"], x_build)
        opti.set_value(mpc_vars["x0_tank"], x_tank)
        opti.set_value(mpc_vars["ta_forecast"], ta_fc)
        opti.set_value(mpc_vars["gh_forecast"], gh_fc)
        opti.set_value(mpc_vars["price_forecast"], price_fc)

        try:
            solution = opti.solve()
            m_dot_hp_cmd = float(solution.value(mpc_vars["M_dot_hp"][0, 0]))
            m_dot_load_cmd = float(solution.value(mpc_vars["M_dot_load"][0, 0]))
        except Exception as exc:
            print(f"Optimization failed at step {k}: {exc}")
            break

        # Plant inputs and derived thermal powers at the current state
        ta_k = float(df["Ta"].iloc[k])
        gh_k = float(df["Gv"].iloc[k])
        price_k = float(df["price_eur_per_kwh"].iloc[k])

        q_hp_cmd = tank_model.q_hp_to_tank_kw_numpy(
            x_tank_c=x_tank,
            m_dot_hp_kg_s=m_dot_hp_cmd,
        )
        q_build_cmd = tank_model.q_tank_to_building_kw_numpy(
            x_tank_c=x_tank,
            m_dot_load_kg_s=m_dot_load_cmd,
        )

        q_hp_max_real = float(hp_capacity_kw(ta_k, tank_model.params.t_hp_supply_c, hp_params))
        cop_real = float(hp_cop(ta_k, tank_model.params.t_hp_supply_c, hp_params))
        p_hp_real_kw = q_hp_cmd / max(cop_real, 1.1)
        cost_step_eur = price_k * p_hp_real_kw * DT_HOURS

        # Building plant update
        x_build = building_model.step_numpy(
            x_k=x_build,
            q_build_kw=q_build_cmd,
            t_out_c=ta_k,
            g_solar=gh_k,
        )

        # Tank plant update
        x_tank = tank_model.step_numpy(
            x_k=x_tank,
            m_dot_hp_kg_s=m_dot_hp_cmd,
            m_dot_load_kg_s=m_dot_load_cmd,
            t_amb_c=ta_k,
        )

        records.append({
            "timestamp": df["timestamp"].iloc[k],
            "Ta_C": ta_k,
            "Gv": gh_k,
            "price_eur_per_kwh": price_k,
            "band": df["band"].iloc[k],
            "m_dot_hp_kg_s": m_dot_hp_cmd,
            "m_dot_load_kg_s": m_dot_load_cmd,
            "Q_hp_kw": q_hp_cmd,
            "Q_build_kw": q_build_cmd,
            "P_hp_kw": p_hp_real_kw,
            "Q_hp_max_kw": q_hp_max_real,
            "COP": cop_real,
            "Ti_C": float(x_build[0]),
            "Te_C": float(x_build[1]),
            "T_tank_bottom_C": float(x_tank[0]),
            "T_tank_middle_C": float(x_tank[1]),
            "T_tank_top_C": float(x_tank[2]),
            "cost_step_eur": cost_step_eur,
        })

    results = pd.DataFrame(records)
    if not results.empty:
        results["cumulative_cost_eur"] = results["cost_step_eur"].cumsum()
        results["cumulative_hp_energy_kwh"] = (results["P_hp_kw"] * DT_HOURS).cumsum()
    return results


results = run_closed_loop_simulation(
    df=df_inputs,
    building_model=building_model,
    tank_model=tank_model,
    hp_params=HP_PARAMS,
    opti=opti,
    mpc_vars=mpc_vars,
)

results.head()


In [ ]:
# -----------------------------
# Diagnostics and plots
# -----------------------------
if results.empty:
    raise RuntimeError("No results were produced. Check the optimization setup.")

print("Simulation summary")
print("-" * 40)
print(f"Time steps simulated: {len(results)}")
print(f"Indoor temperature range [°C]: {results['Ti_C'].min():.2f} -> {results['Ti_C'].max():.2f}")
print(f"Top tank temperature range [°C]: {results['T_tank_top_C'].min():.2f} -> {results['T_tank_top_C'].max():.2f}")
print(f"Total HP electricity use [kWh]: {results['P_hp_kw'].sum() * DT_HOURS:.2f}")
print(f"Total electricity cost [€]: {results['cost_step_eur'].sum():.2f}")

inv_bottom_middle = (results["T_tank_bottom_C"] > results["T_tank_middle_C"]).sum()
inv_middle_top = (results["T_tank_middle_C"] > results["T_tank_top_C"]).sum()
print(f"Bottom-middle inversions: {int(inv_bottom_middle)}")
print(f"Middle-top inversions: {int(inv_middle_top)}")

fig, axes = plt.subplots(6, 1, figsize=(12, 16), sharex=True)

axes[0].plot(results["timestamp"], results["Ti_C"], label="Indoor temperature")
axes[0].axhline(TI_MIN_C, color="k", linestyle="--", alpha=0.4, label="Comfort bounds")
axes[0].axhline(TI_MAX_C, color="k", linestyle="--", alpha=0.4)
axes[0].set_ylabel("Ti [°C]")
axes[0].legend()
axes[0].grid(True)

axes[1].step(results["timestamp"], results["Q_hp_kw"], where="post", label="HP → tank thermal power")
axes[1].step(results["timestamp"], results["Q_build_kw"], where="post", label="Tank → building heat")
axes[1].set_ylabel("Power [kW]")
axes[1].legend()
axes[1].grid(True)

axes[2].step(results["timestamp"], results["m_dot_hp_kg_s"], where="post", label="HP charging flow")
axes[2].step(results["timestamp"], results["m_dot_load_kg_s"], where="post", label="Load-side flow")
axes[2].set_ylabel("m_dot [kg/s]")
axes[2].legend()
axes[2].grid(True)

axes[3].plot(results["timestamp"], results["T_tank_bottom_C"], label="Tank bottom")
axes[3].plot(results["timestamp"], results["T_tank_middle_C"], label="Tank middle")
axes[3].plot(results["timestamp"], results["T_tank_top_C"], label="Tank top")
axes[3].axhline(T_SUPPLY_MIN_C, color="k", linestyle="--", alpha=0.4, label="Useful top-node minimum")
axes[3].set_ylabel("Tank T [°C]")
axes[3].legend()
axes[3].grid(True)

axes[4].step(results["timestamp"], results["price_eur_per_kwh"], where="post", label="Electricity price")
axes[4].set_ylabel("€/kWh")
axes[4].legend()
axes[4].grid(True)

axes[5].plot(results["timestamp"], results["cumulative_cost_eur"], label="Cumulative cost")
axes[5].set_ylabel("€")
axes[5].set_xlabel("Time")
axes[5].legend()
axes[5].grid(True)

plt.tight_layout()
plt.show()


## Remarks for the next iteration

This version moves from the previous power-based tank model to a **flow-based 3-node tank**.

### What is already in place
- the **building RC structure** is preserved,
- the **MPC is storage-sensitive**,
- the **electricity tariff signal** is active,
- the **HP and tank communicate through an equivalent charging flow**,
- the **tank and building communicate through an equivalent load-side flow**,
- useful heat delivery is linked to the **top tank temperature**,
- the eMPC includes a **terminal tank penalty**,
- smooth anti-inversion mixing remains inside both the prediction model and plant update.

### Natural next upgrades
1. Increase the prediction horizon if price-driven charge shifting is too short-sighted.
2. Replace fixed `T_LOAD_RETURN_C` with a return-temperature model from the fan-coil/building side.
3. Replace fixed `T_HP_SUPPLY_C` with an optimized or weather-compensated HP supply temperature.
4. Move from 3-node to 5-node tank representation if the solver remains stable.
5. Add an optional direct **HP → building** bypass if needed in a later architecture.
